# Day 2 — Feature Engineering
Goal: build the tabular + geospatial features every downstream model will use.


In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv('kc_house_data.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.drop_duplicates(subset='id').reset_index(drop=True)
df = df[df['bedrooms'] < 15].reset_index(drop=True)


In [8]:
def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Seattle downtown as reference point
SEATTLE_LAT, SEATTLE_LON = 47.6062, -122.3321
df['dist_to_center_km'] = haversine(df['lat'], df['long'], SEATTLE_LAT, SEATTLE_LON)


In [9]:
df['house_age'] = df['date'].dt.year - df['yr_built']
df['is_renovated'] = (df['yr_renovated'] > 0).astype(int)
df['years_since_renovation'] = np.where(
    df['is_renovated'] == 1,
    df['date'].dt.year - df['yr_renovated'],
    df['house_age']
)
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month

# Ratio features that often help tree models
df['living_lot_ratio'] = df['sqft_living'] / df['sqft_lot'].replace(0, np.nan)
df['basement_flag'] = (df['sqft_basement'] > 0).astype(int)
# EDA only
df['price_per_sqft'] = df['price'] / df['sqft_living']


In [10]:
# Zipcode-level median price as a coarse spatial signal (computed on train split later
# to avoid leakage -- here we just inspect it)
zip_median = df.groupby('zipcode')['price'].median().sort_values(ascending=False)
zip_median.head(10)


,price
zipcode,
98039,1905000.0
98004,1150000.0
98040,993750.0
98112,912500.0
98005,765475.0
98006,760000.0
98119,744975.0
98075,739999.5
98109,736000.0


In [11]:
import folium
m = folium.Map(location=[47.55, -122.2], zoom_start=9, tiles='cartodbpositron')
for zc, row in df.groupby('zipcode').agg(lat=('lat','mean'), long=('long','mean'),
                                          med_price=('price','median')).iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=6,
        popup=f"{zc}: ${row['med_price']:,.0f}",
        color='#3186cc', fill=True, fill_opacity=0.6
    ).add_to(m)
m.save('zipcode_median_price_map.html')
m


# ============================================================
# Feature Engineering
# Spatial Features
# Temporal Features
# Structural Features
# ============================================================

In [12]:
FEATURE_COLS = ['bedrooms','bathrooms','sqft_living','sqft_lot','floors','waterfront',
                'view','condition','grade','sqft_above','sqft_basement','house_age',
                'is_renovated','years_since_renovation','dist_to_center_km',
                'living_lot_ratio','basement_flag','lat','long']

df_model = df[['id','price','zipcode'] + FEATURE_COLS].dropna().reset_index(drop=True)
df_model.to_csv('kc_house_features.csv', index=False)
df_model.describe().T
df_model.head()


,id,price,zipcode,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_above,sqft_basement,house_age,is_renovated,years_since_renovation,dist_to_center_km,living_lot_ratio,basement_flag,lat,long
0,7129300520,221900.0,98178,3,1.00,1180,5650,1.0,0,0,...,1180,0,59,0,59,11.972687,0.208850,0,47.5112,-122.257
1,6414100192,538000.0,98125,3,2.25,2570,7242,2.0,0,0,...,2170,400,63,1,23,12.802819,0.354874,1,47.7210,-122.319
2,5631500400,180000.0,98028,2,1.00,770,10000,1.0,0,0,...,770,0,82,0,82,16.416960,0.077000,0,47.7379,-122.233
3,2487200875,604000.0,98136,4,3.00,1960,5000,1.0,0,0,...,1050,910,49,0,49,10.538233,0.392000,1,47.5208,-122.393
4,1954400510,510000.0,98074,3,2.00,1680,8080,1.0,0,0,...,1680,0,28,0,28,21.553979,0.207921,0,47.6168,-122.045


In [13]:
# interaction features
df['bath_bed_ratio'] = df['bathrooms'] / (df['bedrooms'] + 1)

df['living_per_floor'] = df['sqft_living'] / df['floors']

df['total_rooms'] = df['bedrooms'] + df['bathrooms']

In [14]:
# neighbourhood density
zipcode_density = df['zipcode'].value_counts()

df['zipcode_density'] = df['zipcode'].map(zipcode_density)

## Day 2 Summary

- Engineered spatial features using Haversine distance from Seattle city centre.
- Created temporal features including house age, renovation status, renovation recency, sale month, and sale year.
- Developed structural features including living-area ratio, basement presence, and bathroom-bedroom relationships.
- Analysed neighbourhood-level pricing using zipcode median values and interactive geospatial visualization.
- Built the final feature dataset for downstream machine learning models while preventing target leakage.